# 💻 Análisis Empresarial: Evaluación del Rendimiento de una Empresa de Tecnología

---

## 🎯 Contexto del Problema

Una empresa de tecnología opera en un **mercado altamente competitivo** y necesita evaluar y mejorar su rendimiento financiero y operativo. A diferencia de una empresa tradicional, las empresas tech tienen métricas propias:

- **ARR** (Annual Recurring Revenue): ingresos recurrentes anuales
- **MRR** (Monthly Recurring Revenue): ingresos recurrentes mensuales  
- **Churn Rate**: tasa de cancelación de clientes
- **CAC Payback**: meses para recuperar el costo de adquisición
- **NRR** (Net Revenue Retention): ingresos retenidos netos

Tu objetivo es realizar un **análisis integral** usando metodologías estructuradas.

---

## 🧰 Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# Paleta de colores tech
COLORES = {
    'primario': '#2D46B9',
    'secundario': '#F037A5',
    'exito': '#2ECC71',
    'alerta': '#F39C12',
    'error': '#E74C3C',
    'neutro': '#95A5A6'
}

plt.rcParams['figure.figsize'] = (14, 7)
np.random.seed(42)
print("✅ Setup completado")

---
## 📌 PARTE 0: Generación del Dataset de la Empresa Tech

### Consigna
Crear un dataset que simule 3 años de operación de una empresa de tecnología SaaS (Software as a Service).

### 💡 Pistas
- Una empresa SaaS tiene clientes que pagan **mensualmente** (suscripciones)
- Los ingresos crecen por: nuevos clientes + expansión de clientes actuales
- Los ingresos bajan por: cancelaciones (churn) + downgrade de planes
- Incluí tanto métricas financieras como operativas

In [ ]:
# Dataset mensual de la empresa tech (36 meses = 3 años)

meses = pd.date_range(start='2022-01-01', periods=36, freq='MS')
n_meses = len(meses)

# TU CÓDIGO AQUÍ:
# Creá el dataset con las siguientes variables:

# === CLIENTES ===
# - clientes_inicio: comienza con 500, crece ~8% mensual con volatilidad
# - clientes_nuevos: adquisición mensual (crece pero con meses malos)
# - clientes_churned: cancelaciones (churn_rate ~3% mensual, pero varia)
# - clientes_expandidos: usan más features/plan superior
# - clientes_contratados: reducen plan
# - clientes_fin: inicio + nuevos - churned

# === INGRESOS (en $1000 USD) ===
# - MRR_inicio: MRR al inicio del mes
# - MRR_nuevo: ingresos de nuevos clientes
# - MRR_expansion: ingresos adicionales de clientes existentes
# - MRR_contraccion: pérdida por downgrade
# - MRR_churn: pérdida por cancelaciones
# - MRR_fin: MRR al final del mes
# - ARR: MRR_fin * 12

# === COSTOS ===
# - COGS: Costo de bienes vendidos (hosting, soporte) ~25% de ingresos
# - gasto_RD: Investigación y Desarrollo ~20% de ingresos
# - gasto_marketing: ~15% de ingresos (varia con campañas)
# - gasto_ventas: ~12% de ingresos
# - gasto_admin: ~8% de ingresos

# === MÉTRICAS OPERATIVAS ===
# - CAC: Costo de Adquisición (gasto_marketing+ventas / clientes_nuevos)
# - LTV: Ticket_promedio / churn_rate_mensual
# - NPS_score: varía entre 30 y 70
# - uptime_%: disponibilidad del servicio (99.2% a 99.9%)
# - tickets_soporte: cantidad de tickets abiertos
# - empleados: crece de 45 a 120
# - ingresos_por_empleado: MRR*1000 / empleados

# Mostrá las primeras filas y estadísticas descriptivas


---
## 📌 PARTE 1: Identificación de Objetivos del Análisis

### Consigna
Antes de analizar datos, establecé claramente qué queremos lograr con este análisis.

### 💡 Pistas
- Los objetivos deben ser **SMART**: Específicos, Medibles, Alcanzables, Relevantes, con Tiempo definido
- Clasificalos por área: financiero, operativo, de mercado
- Priorizalos: ¿cuál es el más urgente para la dirección?

In [ ]:
# Definición formal de objetivos del análisis

objetivos = [
    {
        'id': 1,
        'objetivo': '',          # Ej: 'Evaluar la salud financiera actual'
        'pregunta_clave': '',    # Ej: '¿Somos rentables? ¿Cuándo seremos rentables?'
        'metricas': [],          # Ej: ['Margen EBITDA', 'Burn rate', 'Runway']
        'area': '',              # Financiero / Operativo / Mercado / Producto
        'prioridad': ''          # Alta / Media / Baja
    },
    # Agregá al menos 5 objetivos más
]

# TU CÓDIGO AQUÍ:
# 1. Completá al menos 6 objetivos
# 2. Convertí a DataFrame y mostrá ordenado por prioridad
# 3. Identificá qué métricas del dataset necesitamos para cada objetivo

df_objetivos = pd.DataFrame(objetivos)
# print(df_objetivos)


---
## 📌 PARTE 2: Análisis del Mercado Tecnológico

### Consigna
Analizar el entorno competitivo y las tendencias del sector tecnológico.

### 💡 Pistas
- Métricas de la industria SaaS (benchmarks):  
  - Churn mensual saludable: < 2%  
  - Margen bruto SaaS: 65-80%  
  - Rule of 40: crecimiento_ARR% + margen_EBITDA% > 40
  - NRR > 100% indica expansión neta
- Usá la **Rule of 40** como KPI de salud general de la empresa

In [ ]:
# Posicionamiento competitivo en el mercado tech

competidores_tech = pd.DataFrame({
    'Empresa': ['Nuestra_empresa', 'Empresa_A', 'Empresa_B', 'Empresa_C', 'Startup_D', 'Benchmark_Industria'],
    'ARR_M_USD': [],           # Annual Recurring Revenue en millones
    'Crecimiento_ARR_%': [],   # Tasa de crecimiento anual
    'Churn_mensual_%': [],
    'Margen_bruto_%': [],
    'NRR_%': [],               # Net Revenue Retention (>100% = crecimiento)
    'CAC_Payback_meses': [],   # Meses para recuperar CAC
    'LTV_CAC_ratio': [],
    'Empleados': []
})

# TU CÓDIGO AQUÍ:
# 1. Completá la tabla con valores realistas
# 2. Calculá la Rule of 40 para cada empresa:
#    Rule_of_40 = crecimiento_ARR + margen_EBITDA
#    (>40 = excelente, 20-40 = bueno, <20 = preocupante)

# 3. Creá un bubble chart:
#    - Eje X: Churn Rate
#    - Eje Y: Crecimiento ARR
#    - Tamaño burbuja: ARR
#    - Color: Rule of 40
#    Esto muestra visualmente quiénes son los mejores en el sector

# 4. ¿En qué cuadrante está nuestra empresa?
#    Ideal: alto crecimiento + bajo churn


---
## 📌 PARTE 3: Análisis Interno - Modelo de Negocio y Operaciones

### Consigna
Evaluar los recursos internos, costos y eficiencia operativa.

### 💡 Pistas
- El **Burn Rate** indica cuánto dinero gasta la empresa por mes
- El **Runway** indica cuántos meses puede sobrevivir con el dinero actual
- La eficiencia de ventas: `New ARR / Sales & Marketing Spend`
- Cohort analysis: ¿cuándo abandonan los clientes? ¿en qué mes?

In [ ]:
# Análisis de unit economics

def calcular_unit_economics(df):
    """
    Calcula las métricas de unit economics para cada mes.
    """
    ue = pd.DataFrame()
    
    # TU CÓDIGO AQUÍ: calculá cada métrica
    
    # Churn rate mensual
    # ue['churn_rate_%'] = df['clientes_churned'] / df['clientes_inicio'] * 100
    
    # Gross Revenue Retention (GRR)
    # ue['GRR_%'] = 1 - (df['MRR_churn'] + df['MRR_contraccion']) / df['MRR_inicio']
    
    # Net Revenue Retention (NRR) - >100% significa que creces solo con clientes actuales
    # ue['NRR_%'] = (df['MRR_inicio'] + df['MRR_expansion'] - df['MRR_churn'] - df['MRR_contraccion']) / df['MRR_inicio'] * 100
    
    # CAC Payback (meses para recuperar el costo de adquisición)
    # Asumiendo margen bruto del 70%:
    # ue['CAC_payback_meses'] = df['CAC'] / (ticket_promedio_mes * 0.70)
    
    # Eficiencia de ventas
    # ue['eficiencia_ventas'] = df['MRR_nuevo'] * 12 / (df['gasto_marketing'] + df['gasto_ventas'])
    
    # Ingresos por empleado (eficiencia operativa)
    # ue['ingresos_por_empleado'] = ...
    
    return ue

# df_ue = calcular_unit_economics(df)
# Graficá la evolución de cada métrica a lo largo de los 36 meses
# ¿Estamos mejorando? ¿Cuáles métricas empeoran?


In [ ]:
# Análisis de cohortes - Retención de clientes

# Los análisis de cohortes muestran cómo se comportan grupos de clientes
# que empezaron en el mismo período

# TU CÓDIGO AQUÍ:
# Simulá datos de cohortes: para cada cohorte de inicio,
# ¿qué % de clientes sigue activo en el mes 1, 2, 3... 24?

# Ejemplo de estructura:
# cohorte_ene22: [100%, 93%, 88%, 84%, 81%, 79%, ...] (% que sobrevive cada mes)

n_cohortes = 12  # Un año de cohortes mensuales
n_meses_seguimiento = 24

# Generá una matriz de retención realista:
# - La retención en mes 1 suele ser la más alta (90-95%)
# - Luego cae gradualmente
# - Las cohortes más recientes deberían mejorar si implementamos mejoras

# Visualizá como heatmap:
# sns.heatmap(matriz_retencion, annot=True, fmt='.0%', cmap='YlGn')
# Filas = cohortes de inicio
# Columnas = mes de seguimiento (0, 1, 2, ... 23)


---
## 📌 PARTE 4: Balanced Scorecard y KPIs

### Consigna
Implementar el Balanced Scorecard (BSC) con las 4 perspectivas para evaluar el rendimiento global.

### 💡 Pistas
- Las **4 perspectivas del BSC**:
  1. 💰 **Financiera**: ¿Creamos valor para los accionistas?
  2. 🤝 **Clientes**: ¿Cómo nos perciben nuestros clientes?
  3. ⚙️ **Procesos Internos**: ¿En qué procesos debemos ser excelentes?
  4. 📚 **Aprendizaje y Crecimiento**: ¿Podemos mejorar y crear valor?
- Cada perspectiva necesita: objetivo, KPI, meta y resultado actual

In [ ]:
# Balanced Scorecard de la Empresa Tech

bsc = {
    'Financiera': [
        {
            'objetivo': 'Crecer ARR sosteniblemente',
            'kpi': 'Crecimiento ARR anual %',
            'meta': 80,
            'actual': None,    # Calculalo del dataset
            'unidad': '%'
        },
        {
            'objetivo': 'Mejorar eficiencia de costos',
            'kpi': 'Margen bruto %',
            'meta': 72,
            'actual': None,
            'unidad': '%'
        },
        {
            'objetivo': 'Camino a la rentabilidad',
            'kpi': 'EBITDA margin %',
            'meta': -5,        # Negativo OK para una startup en crecimiento
            'actual': None,
            'unidad': '%'
        }
    ],
    'Clientes': [
        {
            'objetivo': 'Reducir abandono de clientes',
            'kpi': 'Churn rate mensual %',
            'meta': 1.5,
            'actual': None,
            'unidad': '%'
        },
        # Completá con más KPIs de clientes
    ],
    'Procesos_Internos': [
        # Completá con KPIs operativos
        # Ej: uptime, tiempo de resolución de tickets, time-to-value
    ],
    'Aprendizaje_y_Crecimiento': [
        # Completá con KPIs de equipo y capacidad
        # Ej: satisfacción de empleados, velocidad de desarrollo, cobertura de tests
    ]
}

# TU CÓDIGO AQUÍ:
# 1. Calculá el valor 'actual' de cada KPI desde el dataset
# 2. Calculá el % de cumplimiento de meta: min(actual/meta, 1.0) * 100
# 3. Asigná un semáforo: verde >85%, amarillo 70-85%, rojo <70%
# 4. Creá una visualización del BSC tipo semáforo o gauge


In [ ]:
# Visualización del Balanced Scorecard

def visualizar_bsc(bsc_data):
    """
    Crea una visualización del BSC con indicadores de semáforo.
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    perspectivas = list(bsc_data.keys())
    
    colores_perspectiva = {
        'Financiera': '#2D46B9',
        'Clientes': '#F037A5', 
        'Procesos_Internos': '#2ECC71',
        'Aprendizaje_y_Crecimiento': '#F39C12'
    }
    
    # TU CÓDIGO AQUÍ:
    # Para cada perspectiva, creá un panel que muestre:
    # - Nombre de la perspectiva con su color
    # - Cada KPI con: nombre, valor actual, meta, semáforo (●)
    # - Un gauge o barra de progreso
    
    pass

# visualizar_bsc(bsc)


---
## 📌 PARTE 5: Análisis FODA Estratégico

### Consigna
Realizar un análisis FODA completo con **estrategias cruzadas**.

### 💡 Pistas
- Las estrategias cruzadas combinan factores internos y externos:
  - **FO**: Usar Fortalezas para aprovechar Oportunidades
  - **DO**: Superar Debilidades usando Oportunidades
  - **FA**: Usar Fortalezas para mitigar Amenazas
  - **DA**: Minimizar Debilidades y evitar Amenazas
- Cada elemento del FODA debe estar **sustentado en datos** de los análisis previos

In [ ]:
# Análisis FODA con evidencia cuantitativa

foda_tech = {
    'Fortalezas': [
        # Basate en los análisis anteriores
        # Ej: {'factor': 'NRR > 110%', 'evidencia': 'Clientes actuales generan crecimiento neto'}
    ],
    'Oportunidades': [
        # Ej: {'factor': 'Mercado SaaS crece 18% anual', 'evidencia': 'Benchmark industria'}
    ],
    'Debilidades': [
        # Ej: {'factor': 'CAC Payback > 18 meses', 'evidencia': 'Calculado en Parte 3'}
    ],
    'Amenazas': [
        # Ej: {'factor': 'Competidores con 10x más capital', 'evidencia': 'Análisis mercado Parte 2'}
    ]
}

# Estrategias cruzadas
estrategias = {
    'FO': [],  # Fortalezas + Oportunidades: estrategias de crecimiento
    'DO': [],  # Debilidades + Oportunidades: estrategias de mejora
    'FA': [],  # Fortalezas + Amenazas: estrategias defensivas
    'DA': []   # Debilidades + Amenazas: estrategias de supervivencia
}

# TU CÓDIGO AQUÍ:
# 1. Completá el FODA con al menos 4 items por cuadrante
# 2. Completá las estrategias cruzadas (al menos 2 por combinación)
# 3. Imprimí el FODA de forma visual

# Bonus: Creá una visualización de la matriz FODA como tabla coloreada
# usando matplotlib con colores por cuadrante


---
## 📌 PARTE 6: Análisis Financiero Detallado

### Consigna
Preparar el informe financiero completo de la empresa.

### 💡 Pistas
- Para empresas SaaS, el **P&L (Estado de Resultados)** sigue un formato específico
- El **Burn Rate** y **Runway** son críticos para startups
- Los inversores tech miran: ARR, Net Burn, Gross Margin, NRR
- Proyectar hacia la **rentabilidad** (breakeven)

In [ ]:
# Estado de Resultados (P&L) - Formato SaaS

def generar_pl_saas(df):
    """
    Genera el P&L en formato SaaS por trimestre.
    """
    # Agrupá por trimestre
    # TU CÓDIGO AQUÍ:
    
    pl = pd.DataFrame(columns=['Q1-22', 'Q2-22', 'Q3-22', 'Q4-22',
                                 'Q1-23', 'Q2-23', 'Q3-23', 'Q4-23',
                                 'Q1-24', 'Q2-24', 'Q3-24', 'Q4-24'])
    
    # Completá las filas:
    # INGRESOS:
    # Revenue (MRR * 1000 * 3 meses)
    
    # COSTO DE REVENUE:
    # (-) COGS (hosting, soporte técnico)
    # = Gross Profit
    # Gross Margin %
    
    # GASTOS OPERATIVOS:
    # (-) R&D
    # (-) Sales & Marketing
    # (-) G&A (General & Administrative)
    # = Total OpEx
    
    # RESULTADOS:
    # = Operating Income (EBIT)
    # EBIT Margin %
    # = EBITDA (aproximado)
    # EBITDA Margin %
    
    # Net Burn Rate (si negativo)
    # Runway (asumiendo $5M en caja inicial)
    
    return pl

# df_pl = generar_pl_saas(df)
# Mostrá el P&L con formato y porcentajes


In [ ]:
# Proyección hacia el breakeven (rentabilidad)

# TU CÓDIGO AQUÍ:
# Usando el modelo predictivo, proyectá:
# 1. ¿En qué mes la empresa alcanza Gross Margin > 65%?
# 2. ¿En qué mes alcanza breakeven (EBITDA = 0)?
# 3. ¿Cuál es el ARR en ese punto?

# Asumí que:
# - El crecimiento de ingresos continúa al ritmo actual
# - Los costos crecen más lento que los ingresos (economías de escala)
# - R&D se mantiene como % de ingresos
# - Sales & Marketing baja como % a medida que crece el NRR

# Graficá:
# - Ingresos vs. Costos totales (área sombreada entre ellas = pérdida/ganancia)
# - Línea vertical en el punto de breakeven
# - Anotación con la fecha y ARR proyectado del breakeven

# Caja disponible (asumí $5M inicial):
caja_inicial = 5_000  # En $1000 USD
# caja_acumulada = caja_inicial + cumsum(net_burn)
# ¿Tenemos suficiente runway para llegar al breakeven?


In [ ]:
# Ratios financieros tech

def calcular_ratios_tech(df, caja_actual):
    """Calcula ratios financieros relevantes para empresas tech/SaaS."""
    
    ultimo_mes = df.iloc[-1]
    ultimos_12m = df.tail(12)
    
    ratios = {}
    
    # TU CÓDIGO AQUÍ:
    
    # Gross Margin %
    # ratios['Margen_Bruto_%'] = ...
    
    # Rule of 40
    # crecimiento_anual = (ultimo_mes['ARR'] / df.iloc[-13]['ARR'] - 1) * 100
    # ratios['Rule_of_40'] = crecimiento_anual + margen_ebitda
    # ratios['Evaluacion_Rule40'] = 'Excelente' if ratios['Rule_of_40'] > 40 else ...
    
    # NRR (Net Revenue Retention)
    # ratios['NRR_%'] = ...
    
    # Eficiencia de ventas (Magic Number)
    # Magic Number = (cambio en ARR * margen_bruto) / gasto_s&m_trimestre_anterior
    # ratios['Magic_Number'] = ...
    
    # Burn Multiple (cuánto gastamos por cada $1 de ARR nuevo)
    # Ideal < 1.0
    # ratios['Burn_Multiple'] = net_burn_anual / nuevo_arr_anual
    
    # Runway
    # burn_mensual_promedio = ...
    # ratios['Runway_meses'] = caja_actual / burn_mensual_promedio
    
    return pd.Series(ratios)

# ratios = calcular_ratios_tech(df, caja_actual=caja_inicial)
# Mostrá con interpretación de cada ratio


---
## 📌 PARTE 7: Análisis de Cadena de Valor

### Consigna
Examinar las actividades de la empresa para identificar dónde se crea y dónde se destruye valor.

### 💡 Pistas
- La cadena de valor de Porter tiene actividades **primarias** y **de soporte**
- Para tech: foco en desarrollo de producto, ventas y soporte al cliente
- Identificá el **cuello de botella** principal en el proceso
- Calculá el costo y tiempo de cada etapa del customer journey

In [ ]:
# Customer Journey - Análisis de tiempos y costos por etapa

customer_journey = pd.DataFrame({
    'Etapa': [
        'Lead Generation',     # Atracción de prospectos
        'MQL → SQL',           # Calificación de leads
        'Demo/Trial',          # Evaluación del producto
        'Propuesta/Negociación',
        'Onboarding',          # Implementación
        'Adopción',            # Uso activo
        'Expansión',           # Upsell/cross-sell
        'Retención/Renovación'
    ],
    'Duracion_promedio_dias': [],   # Tiempo promedio en esta etapa
    'Tasa_conversion_%': [],        # % que avanza a la siguiente etapa
    'Costo_por_cuenta_USD': [],     # Costo de gestionar esta etapa
    'Equipo_responsable': [],       # Marketing / Ventas / CS / Producto
    'Principal_obstaculo': []       # ¿Dónde se pierden más leads?
})

# TU CÓDIGO AQUÍ:
# 1. Completá con datos realistas del sector SaaS
# 2. Calculá el costo total del CAC sumando todas las etapas pre-cierre
# 3. Graficá un funnel de conversión:
#    - Empezando con 10,000 leads en la primera etapa
#    - Aplicando las tasas de conversión
#    - ¿Cuántos clientes resultan?
#    - ¿En qué etapa se pierde más?

# Pista para funnel chart:
# Usá barras horizontales con el ancho proporcional a los prospectos
# decrece a medida que baja el funnel


---
## 📌 PARTE 8: Análisis Predictivo - Proyección de ARR y Clientes

### Consigna
Modelar el crecimiento futuro de la empresa bajo diferentes escenarios de inversión.

### 💡 Pistas
- Modelá el crecimiento como función del gasto en adquisición y la retención
- Escenarios: Base, Boost de Marketing, Reducción de Churn, Ambos
- Usá simulación de Monte Carlo para mostrar rango de posibles outcomes

In [ ]:
# Modelo de crecimiento SaaS simplificado

def simular_crecimiento_saas(
    mrr_inicial,
    clientes_iniciales,
    tasa_nuevos_clientes_mensual,
    ticket_promedio,
    churn_mensual,
    expansion_mensual,
    n_meses=24
):
    """
    Simula el crecimiento de MRR y clientes durante n_meses.
    """
    resultados = []
    mrr = mrr_inicial
    clientes = clientes_iniciales
    
    for mes in range(n_meses):
        # TU CÓDIGO AQUÍ:
        # 1. Calculá clientes nuevos (tasa_nuevos_clientes_mensual * clientes actuales)
        # 2. Calculá clientes perdidos (churn_mensual * clientes)
        # 3. Actualizá clientes
        # 4. Calculá MRR_nuevo, MRR_churn, MRR_expansion
        # 5. Actualizá MRR
        
        resultados.append({
            'mes': mes + 1,
            'clientes': clientes,
            'MRR': mrr,
            'ARR': mrr * 12
        })
    
    return pd.DataFrame(resultados)


# Escenario Base: continúa como hasta ahora
# base = simular_crecimiento_saas(
#     mrr_inicial=df['MRR_fin'].iloc[-1],
#     clientes_iniciales=df['clientes_fin'].iloc[-1],
#     tasa_nuevos_clientes_mensual=0.08,
#     ticket_promedio=df['MRR_fin'].iloc[-1] / df['clientes_fin'].iloc[-1],
#     churn_mensual=0.025,
#     expansion_mensual=0.005
# )

# Escenario Boost: +50% en adquisición + reducción churn a 1.5%
# Escenario Foco Retención: solo reducir churn a 1%
# Escenario Combinado: todo junto

# Graficá los 4 escenarios de ARR a 24 meses


In [ ]:
# Simulación de Monte Carlo para el ARR

# TU CÓDIGO AQUÍ:
# La idea es simular 1000 posibles trayectorias de crecimiento,
# variando aleatoriamente los parámetros dentro de rangos razonables

n_simulaciones = 1000
n_meses_proyeccion = 24
resultados_monte_carlo = []

# Para cada simulación:
# 1. Samplea aleatoriamente:
#    - tasa_nuevos: normal(media=0.08, std=0.02)
#    - churn: normal(media=0.025, std=0.008)
#    - expansion: normal(media=0.005, std=0.002)
# 2. Simulá el crecimiento con simular_crecimiento_saas()
# 3. Guardá el ARR final

# Graficá:
# - Histograma del ARR en el mes 24
# - Percentiles 10, 50 (mediana), 90
# - Probabilidad de superar ciertos hitos (ej: $10M ARR)
# - Distribución de trayectorias (muchas líneas grises semitransparentes)


---
## 📌 PARTE 9: Dashboard Ejecutivo Integral

### Consigna
Crear el dashboard final que integre todos los análisis para la presentación a directivos/inversores.

### 💡 Pistas
- Un pitch deck de tech incluye: ARR, growth rate, NRR, Gross Margin, burn, runway
- Usá el estilo visual de las empresas tech: oscuro, limpio, datos grandes
- Cada gráfico debe poder leerse en 3 segundos

In [ ]:
# Dashboard Ejecutivo - Estilo Empresa Tech

fig = plt.figure(figsize=(22, 14), facecolor='#0F1117')
gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Colores del tema dark
COLOR_FONDO = '#0F1117'
COLOR_PANEL = '#1A1D2B'
COLOR_TEXTO = '#FFFFFF'
COLOR_ACENTO = '#2D46B9'
COLOR_POSITIVO = '#2ECC71'
COLOR_NEGATIVO = '#E74C3C'

# TU CÓDIGO AQUÍ - Creá los siguientes paneles:

# Panel 1 (gs[0, :2]): ARR y crecimiento - gráfico de área
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor(COLOR_PANEL)
# Completá...

# Panel 2 (gs[0, 2]): MRR Waterfall del último mes
# (Nuevo - Expansión - Churn - Contracción = Net New MRR)

# Panel 3 (gs[0, 3]): Rule of 40 gauge

# Panel 4 (gs[1, :2]): Evolución de Churn y NRR

# Panel 5 (gs[1, 2]): Cohorte de retención (heatmap)

# Panel 6 (gs[1, 3]): CAC y LTV evolución

# Panel 7 (gs[2, :2]): Proyección de escenarios ARR

# Panel 8 (gs[2, 2:]): BSC semáforo

# Título del dashboard
fig.suptitle('TECH COMPANY - EXECUTIVE DASHBOARD\nBoard Review Q4 2024',
             fontsize=20, fontweight='bold', color=COLOR_TEXTO, y=1.01)

plt.savefig('dashboard_tech_company.png', dpi=150, bbox_inches='tight',
            facecolor=COLOR_FONDO)
plt.show()


---
## 📌 PARTE 10: Roadmap Estratégico y Plan de Mejora

### Consigna
Definir el plan de mejora para los próximos 12 meses basado en todos los análisis.

### 💡 Pistas
- Priorizá iniciativas usando el **RICE framework**: Reach × Impact × Confidence / Effort
- Las iniciativas de producto, ventas y retención tienen diferentes horizontes temporales
- El objetivo financiero debe ser claro: ¿breakeven?, ¿levantar ronda de inversión?

In [ ]:
# Priorización RICE de iniciativas

def calcular_rice(reach, impact, confidence, effort):
    """
    Calcula el score RICE para priorización.
    RICE = (Reach × Impact × Confidence) / Effort
    
    Args:
        reach: # de clientes/usuarios afectados en 3 meses
        impact: escala 0.25 (mínimo) a 3 (transformacional)
        confidence: % de confianza en estimaciones (0-100)
        effort: personas×semana necesarias
    """
    return (reach * impact * confidence/100) / effort


iniciativas = pd.DataFrame({
    'Iniciativa': [
        # Completá con las iniciativas identificadas en los análisis
        # Ej: 'Implementar programa de Customer Success'
        # 'Rediseñar proceso de onboarding'
        # 'Lanzar plan de precios annual con descuento'
    ],
    'Reach': [],           # Nro. de clientes/usuarios
    'Impact': [],          # 0.25 / 0.5 / 1 / 2 / 3
    'Confidence_%': [],    # 0 a 100
    'Effort_pw': [],       # Person-weeks
    'Categoria': [],       # Retención / Adquisición / Producto / Proceso
    'Horizonte': [],       # 0-3m / 3-6m / 6-12m
})

# TU CÓDIGO AQUÍ:
# 1. Completá con al menos 10 iniciativas
# 2. Calculá el RICE score para cada una
# 3. Ordená de mayor a menor
# 4. Graficá un scatter: Effort vs. Impact, tamaño = RICE, color = categoría


In [ ]:
# Roadmap de 12 meses con hitos financieros

# TU CÓDIGO AQUÍ:
# Creá un roadmap que incluya:
# 1. Las iniciativas priorizadas por RICE, agrupadas por horizonte temporal
# 2. Hitos financieros clave:
#    - $XM ARR
#    - Reducir churn a Y%
#    - Alcanzar Gross Margin de Z%
# 3. Puntos de decisión (gates): momentos donde se decide si continuar o pivotar

# Formato Gantt con dos vistas:
# Vista 1: Iniciativas por categoría (colores)
# Vista 2: Hitos de ARR como puntos en el tiempo

# También incluí:
# - Headcount plan: cuántos empleados necesitamos contratar y cuándo
# - Presupuesto requerido por trimestre
# - Fecha proyectada de breakeven


---
## 📌 PARTE 11: Informe Ejecutivo para Board/Inversores

### Consigna
Redactá el informe ejecutivo completo. Debe poder leerse en 5 minutos y tomar decisiones en base a él.

## INFORME EJECUTIVO - BOARD REVIEW

**Empresa:** [Nombre]  
**Período analizado:** Enero 2022 - Diciembre 2024  
**Preparado por:** Equipo de Analytics

---

### 🚀 Situación Actual (completá con datos reales del análisis)

| Métrica | Actual | Meta | Estado |
|---------|--------|------|--------|
| ARR | $XM | $YM | 🟡 |
| Crecimiento ARR | X% | 80% | 🟢 |
| Churn Mensual | X% | <2% | 🔴 |
| NRR | X% | >110% | |
| Gross Margin | X% | >70% | |
| Rule of 40 | X | >40 | |
| Runway | X meses | >18m | |

### 💡 Principales Hallazgos

1. **[Hallazgo 1]** — [Sustentado en datos del análisis]
2. **[Hallazgo 2]** — [Con evidencia cuantitativa]
3. **[Hallazgo 3]** — [Con evidencia cuantitativa]

### 🎯 Recomendaciones Estratégicas

**Prioridad 1 - [Nombre]** (RICE Score: X)
- Qué: [Descripción]
- Por qué: [Evidencia del análisis]
- Impacto esperado: [En ARR, clientes, churn]
- Inversión: [$X en Y semanas]

**Prioridad 2 - [Nombre]**...

### 📈 Proyección a 12 meses

- **Escenario Base:** ARR de $XM, X clientes
- **Escenario Optimista:** ARR de $YM, breakeven en mes Z
- **Próxima ronda de inversión:** [Recomendación]

### ⚡ Decisiones requeridas del Board
1. ¿Aprobamos presupuesto adicional de $X para [iniciativa]?
2. ¿Continuamos con la estrategia actual o pivotamos a [alternativa]?
3. ¿Cuándo iniciar proceso de fundraising?

---
## 🏆 Checklist de Entrega

- [ ] Dataset de 36 meses generado con métricas SaaS coherentes
- [ ] Análisis competitivo con Rule of 40 y bubble chart
- [ ] Unit economics calculados con evolución temporal
- [ ] Análisis de cohortes con heatmap de retención
- [ ] BSC completo con las 4 perspectivas y semáforos
- [ ] FODA con estrategias cruzadas sustentadas en datos
- [ ] P&L trimestral con proyección a breakeven
- [ ] Ratios tech calculados con interpretación
- [ ] Funnel de conversión del customer journey
- [ ] 3+ modelos predictivos comparados
- [ ] Simulación Monte Carlo con percentiles
- [ ] Dashboard ejecutivo dark mode exportado
- [ ] Priorización RICE con scatter chart
- [ ] Roadmap 12 meses con hitos financieros
- [ ] Informe ejecutivo completo

---
*Ejercicio desarrollado como parte del programa de Análisis de Negocios y Ciencia de Datos.*